# RAVDESS Dataset Video Preprocessing and Timesformer Model

This notebook demonstrates the preprocessing of the RAVDESS (Ryerson Audio-Visual Database of Emotional Speech and Song) dataset, specifically focusing on the video modality. It involves downloading the dataset from Kaggle, filtering video files based on specific criteria, extracting frames from these videos using `ffmpeg`, and preparing a custom PyTorch `Dataset` for use with the Timesformer model. The notebook also shows how to load a pre-trained Timesformer model for video classification.

# Pre-processing videos

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("orvile/ravdess-dataset")

print("Path to dataset files:", path)

100%|██████████| 23.9G/23.9G [04:59<00:00, 85.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/orvile/ravdess-dataset/versions/1


Modality (01 = full-AV, 02 = video-only, 03 = audio-only).

Vocal channel (01 = speech, 02 = song).

Emotion (01 = neutral, 02 = calm, 03 = happy, 04 = sad, 05 = angry, 06 = fearful, 07 = disgust, 08 = surprised).

Emotional intensity (01 = normal, 02 = strong). NOTE: There is no strong intensity for the 'neutral' emotion.

Statement (01 = "Kids are talking by the door", 02 = "Dogs are sitting by the door").

Repetition (01 = 1st repetition, 02 = 2nd repetition).

Actor (01 to 24. Odd numbered actors are male, even numbered actors are female).

In [ ]:
from pathlib import Path

# List contents of the downloaded Kaggle dataset root directory
print("Contents of the Kaggle dataset root directory:")
for item in Path(path).iterdir():
    print(item.name)

Contents of the Kaggle dataset root directory:
Video_Speech_Actor_04
Video_Speech_Actor_08
Video_Speech_Actor_12
Video_Song_Actor_13
Video_Song_Actor_22
Video_Song_Actor_07
Video_Speech_Actor_02
Video_Speech_Actor_01
Video_Song_Actor_01
Video_Speech_Actor_19
Video_Song_Actor_02
Video_Song_Actor_05
Video_Song_Actor_24
Video_Song_Actor_23
Video_Speech_Actor_16
Video_Speech_Actor_03
Video_Speech_Actor_13
Video_Song_Actor_12
Video_Speech_Actor_09
Video_Speech_Actor_18
Video_Song_Actor_16
Video_Speech_Actor_20
Video_Song_Actor_11
Video_Song_Actor_14
Video_Song_Actor_15
Video_Song_Actor_21
Video_Song_Actor_03
Video_Speech_Actor_11
Video_Speech_Actor_07
Video_Speech_Actor_21
Video_Song_Actor_09
Video_Speech_Actor_05
Video_Speech_Actor_14
Video_Song_Actor_10
Video_Speech_Actor_06
Video_Song_Actor_08
Video_Speech_Actor_24
Video_Song_Actor_04
Video_Speech_Actor_10
Video_Speech_Actor_23
Audio_Song_Actors_01-24
Video_Song_Actor_17
Video_Speech_Actor_22
Video_Speech_Actor_15
Video_Speech_Actor_17
V

In [ ]:
# from pathlib import Path
# video_path = path+"/archive (18)" + "/Multimodel_Dataset" + "/Video_Dataset" + "/Video_Dataset"

In [ ]:
video_path = Path(path)

In [ ]:
video_path.dtype

AttributeError: 'PosixPath' object has no attribute 'dtype'

In [ ]:
# obtain the frames for all the videos present in video_path
import glob
import os

# video_path = "/path/to/your/directory"

all_videos = list(glob.glob(
    os.path.join(video_path, "**", "*.mp4"),
    recursive=True
))

all_videos2 = all_videos
keyword = "Video_Speech_Actor"

filtered_videos = [
    video for video in all_videos2
    if keyword in os.path.dirname(video)
]

print(len(filtered_videos))
print(len(all_videos))
print(len(all_videos2))

2880
4904
4904


In [ ]:
filtered_videos[2]

'/root/.cache/kagglehub/datasets/orvile/ravdess-dataset/versions/1/Video_Speech_Actor_04/Actor_04/02-01-08-01-01-02-04.mp4'

In [ ]:
import os
import re
import zipfile
from google.colab import files

# 1. Define target directory
search_directory = filtered_videos

# 2. Build the exact RegEx pattern
# \d{2} matches two digits, but character sets like [0-1] restrict the numerical values
pattern = re.compile(
    r"^01-01-"                         # modality(full AV - Vocal Channel(speech))
    r"(0[1-8])-"                       # emotion
    r"(0[1-2])-"                       # 01 or 02(emotional intensity)
    r"(0[1-2])-"                       # 01 or 02(statement)
    # r"(0[1-2])-"                     # 01 or 02(repetition)
    r"01-"                             # Followed by 01-(reprtition)
    r"(0[1-9]|1[0-9]|2[0-4])"          # Any double digit from 01 to 24(actor id)
    r"\.mp4$"                          # Ends exactly with .mp4
)

# 3. Scan the directory for matching files
matching_files = []
for file_path in search_directory:
    file_name = os.path.basename(file_path)
    if pattern.match(file_name):
        matching_files.append(file_path)

# 4. Compile into a ZIP archive and download
# if len(matching_files) > 0:
#     zip_filename = "filtered_videos.zip"

#     with zipfile.ZipFile(zip_filename, 'w') as zipf:
#         for file in matching_files:
#             zipf.write(file, arcname=os.path.basename(file))
#             print(f"Matched & Packaged: {os.path.basename(file)}")

#     print(f"\nSuccessfully compiled {len(matching_files)} video files.")
#     files.download(zip_filename)
# else:
#     print("No files matched the specified structural pattern.")
len(matching_files)

720

In [ ]:
# ! ffmpeg -i sample.mp4 -vf fps=30 frames/frame%04d.png

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# extract frames from all_videos and store them to drive
SAVE_FFMPEG_FRAMES = "/content/drive/MyDrive/Timesformer"

import cv2
import os
import subprocess
i=0
for vid in matching_files:
    print(f"Started {i}:")
    video_name = os.path.splitext(os.path.basename(vid))[0]
    output_dir = os.path.join(SAVE_FFMPEG_FRAMES, video_name)

    os.makedirs(output_dir, exist_ok=True)

    subprocess.run(
        [
            "ffmpeg",
            "-i", vid,
            "-vf", "fps=30",
            f"{output_dir}/frame%04d.png"
        ],
        check=True
    )
    i=i+1

Started 0:
Started 1:
Started 2:
Started 3:
Started 4:
Started 5:
Started 6:
Started 7:
Started 8:
Started 9:
Started 10:
Started 11:
Started 12:
Started 13:
Started 14:
Started 15:
Started 16:
Started 17:
Started 18:
Started 19:
Started 20:
Started 21:
Started 22:
Started 23:
Started 24:
Started 25:
Started 26:
Started 27:
Started 28:
Started 29:
Started 30:
Started 31:
Started 32:
Started 33:
Started 34:
Started 35:
Started 36:
Started 37:
Started 38:
Started 39:
Started 40:
Started 41:
Started 42:
Started 43:
Started 44:
Started 45:
Started 46:
Started 47:
Started 48:
Started 49:
Started 50:
Started 51:
Started 52:
Started 53:
Started 54:
Started 55:
Started 56:
Started 57:
Started 58:
Started 59:
Started 60:
Started 61:
Started 62:
Started 63:
Started 64:
Started 65:
Started 66:
Started 67:
Started 68:
Started 69:
Started 70:
Started 71:
Started 72:
Started 73:
Started 74:
Started 75:
Started 76:
Started 77:
Started 78:
Started 79:
Started 80:
Started 81:
Started 82:
Started 83:
St

KeyboardInterrupt: 

In [ ]:
import os
import glob
import torch
from torchvision.io import read_image

In [ ]:
class VideoDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.video_paths = glob.glob(os.path.join(root_dir, '*.mp4'))
        self.label_map = {'1': 1, '2': 2, '3': 3,'4': 4, '5': 5, '6': 6,'7':7,'8': 8}

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        frames = []
        for frame_path in sorted(glob.glob(os.path.join(video_path, '*.png'))):
            frame = read_image(frame_path)
            frames.append(frame)
        frames = torch.stack(frames)
        label = self.label_map[os.path.basename(video_path).split('_')[0]]
        label = os.path.basename(video_path[7])
        return frames, label

In [ ]:
!pip install timesformer

ERROR: Could not find a version that satisfies the requirement timesformer (from versions: none)
ERROR: No matching distribution found for timesformer


In [ ]:
import torch
from torch import nn
from transformers import TimesformerForVideoClassification
# from timesformer.models.video_transformer import TimeSformer



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/22.7k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/486M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/249 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/486M [00:00<?, ?B/s]

In [ ]:
model = TimesformerForVideoClassification.from_pretrained(
    "facebook/timesformer-base-finetuned-k400"

)

In [ ]:
# create a metadata file

In [ ]:
# class VideoClassifier(nn.Module):
#     def __init__(self, num_classes):
#         super().__init__()
#         self.timesformer = TimesformerModel(
#             dim = 512,
#             depth = 12,
#             heads = 8,
#             clip_len = 16,
#             num_classes = num_classes
#         )

#     def forward(self, x):
#         return self.timesformer(x)